
# MiniMax H3 Comfy 2xT4 Generator (no WanGP)

- ComfyUI 0.35+ with native MiniMax H3
- H3 MultiStream node: splits the H3 transformer across both Kaggle T4s
- exchange=host, exchange_chunks=8 for PCIe cards
- Assets from Comfy-Org/MiniMax-H3
- Smoke test: official `video_minimax_h3_t2v.json` with H3MultiStream inserted into the model path


In [ ]:

import os
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN","")
if HF_TOKEN: os.environ["HF_TOKEN"]=HF_TOKEN
os.environ.setdefault("HF_HUB_DISABLE_XET","1")
os.environ.setdefault("HF_HUB_DISABLE_HF_TRANSFER","1")
os.environ.setdefault("HF_HOME","/tmp/hf")
os.environ.setdefault("TORCH_HOME","/tmp/torch")
os.environ.setdefault("XDG_CACHE_HOME","/tmp/xdg_cache")
print("env ready")

for d in ("/tmp/hf","/tmp/torch","/tmp/xdg_cache"):
    os.makedirs(d, exist_ok=True)


In [ ]:

import shutil,subprocess,psutil,torch
print("torch",torch.__version__,torch.version.cuda)
print("device_count",torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i); f,t=torch.cuda.mem_get_info(i)
    print(f"GPU{i} {p.name} {t/2**30:.1f}GB free {f/2**30:.1f}GB")
ram=psutil.virtual_memory()
print(f"RAM {ram.total/2**30:.1f}G / {ram.available/2**30:.1f}G avail")
print("tmp",shutil.disk_usage("/tmp").free/2**30,"GiB free")
print(subprocess.run(["nvidia-smi","topo","-m"],capture_output=True,text=True).stdout)


In [ ]:

import subprocess,sys,os,shutil
CTEST="/tmp/ComfyUI"
if not os.path.isdir(CTEST):
    subprocess.run(["git","clone","--depth","1","https://github.com/Comfy-Org/ComfyUI.git",CTEST],check=True)
subprocess.run([sys.executable,"-m","pip","install","-r",os.path.join(CTEST,"requirements.txt")],check=True)
cn=os.path.join(CTEST,"custom_nodes","ComfyUI-H3-MultiStream")
if not os.path.isdir(cn):
    subprocess.run(["git","clone","--depth","1","https://github.com/martonsagi/Comfy-H3-MultiStream.git",cn],check=True)
print("ComfyUI at",CTEST)


In [ ]:

import subprocess,sys
subprocess.run([sys.executable,"-m","pip","install","-U","huggingface_hub"],check=True)
subprocess.run([sys.executable,"-m","huggingface_hub.cli.download","Comfy-Org/MiniMax-H3",
    "--include","diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors","text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors","vae/minimax_h3_video_vae_fp16.safetensors","vae/minimax_h3_audio_vae_fp32.safetensors",
    "--local-dir","/tmp/ComfyUI/models"],check=True)
subprocess.run(["curl","-L","-sSf","-o","/tmp/h3_t2v.json","https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_t2v.json"],check=True)
print("assets ok")


In [ ]:

import json, pathlib, urllib.request
src = "/tmp/h3_t2v.json"
wf = json.load(open(src))
if isinstance(wf, dict) and "nodes" in wf:
    nodes=wf["nodes"]; links=wf.get("links", [])
    guider=[n for n in nodes if n.get("type")=="BasicGuider"]
    if not guider:
        print("no BasicGuider, skip patch")
    else:
        basic=guider[0]
        model_in=[i for i in basic["inputs"] if i["name"]=="model"][0]
        lid=model_in["link"]
        old=[L for L in links if L[0]==lid]
        if old:
            nid = max((n["id"] for n in nodes), default=1)+1
            h3={"id":nid,"type":"H3MultiStream","pos":[-1000,4700],"size":[420,240],"flags":{},"order":1,"mode":0,
                "inputs":[{"name":"model","type":"MODEL","link":lid}],
                "outputs":[{"name":"model","type":"MODEL","links":[]}],
                "widgets_values":[True,-1,"host",8,False,"keep",False,2.0,False,True,0.0]}
            basic["inputs"][[i["name"] for i in basic["inputs"]].index("model")]["link"]=old[0][0]+1000
            links.append([old[0][0]+1000, old[0][1], old[0][2], nid,0,"MODEL"])
            old[0][3]=nid
            old[0][4]=0
            h3["outputs"][0]["links"]=[old[0][0]+1000]
            nodes.append(h3)
            wf["nodes"]=nodes
            wf["links"]=links
            with open("/tmp/h3_t2v_ms.json","w") as f: json.dump(wf,f,indent=1)
            print("patched workflow saved /tmp/h3_t2v_ms.json")
        else:
            print("no old link found; using original")
            pathlib.Path("/tmp/h3_t2v_ms.json").write_text(json.dumps(wf),encoding="utf-8")


In [ ]:

import subprocess,sys,os,time,json
comfy="/tmp/ComfyUI"
pr=subprocess.Popen([sys.executable,os.path.join(comfy,"main.py"),"--listen","127.0.0.1","--port","8188","--preview-method","latent2rgb"],cwd=comfy,stdout=open("/tmp/comfy.log","w"),stderr=subprocess.STDOUT)
print("comfy pid",pr.pid)
for i in range(60):
    try:
        r=subprocess.run(["curl","-fs","http://127.0.0.1:8188/history"],capture_output=True) if True else None
        import urllib.request as ur
        ur.urlopen("http://127.0.0.1:8188/history",timeout=3)
        print("comfy ready"); break
    except Exception: time.sleep(5)


In [ ]:

import json,time,urllib.request,uuid,os
pf = "/tmp/h3_t2v_ms.json" if os.path.exists("/tmp/h3_t2v_ms.json") else "/tmp/h3_t2v.json"
wf = json.load(open(pf))
prompt = {"prompt": wf}
url = "http://127.0.0.1:8188/prompt"
req = urllib.request.Request(url, data=json.dumps(prompt).encode(), headers={"Content-Type":"application/json"})
try:
    resp = urllib.request.urlopen(req, timeout=30).read().decode()
    print("submitted:", resp)
except Exception as e:
    print("submit failed:", e)
